# STEP 04C — Uniform COCO AP control for four pediatric caries detection pipelines

**One evaluation protocol, same 20 pre-existing cross-validation checkpoints, no training.** This is a secondary **evaluator-harmonization control** designed to address the reviewers' comparability objection. It does *not* isolate the effect of detector architecture because training optimizers, preprocessing, augmentation and checkpoint-selection rules differed between pipelines.

**Previously audited checkpoint identity:** SHA-256 hashes transcribed directly from the successfully completed `Step_04B_Checkpoint_Replay_20261024T164730Z_Results.zip`; all 20 are required to match before inference. The original source-defined radiograph labels are used without amendment. This analysis must not be described as expert-corrected GT.

**Uniform scoring specification:** COCO-style bounding-box mAP@0.50 and mAP@0.50:0.95 using **TorchMetrics MeanAveragePrecision** (`pycocotools`, 101 recall points, at most 100 predictions/image), labels remapped to class 0 for all detectors. All model predictions are evaluated at a **0.001 score floor**, maximum 100 boxes/image and target inference NMS IoU 0.70 where applicable. Image preprocessing remains model-native (Ultralytics resized to 1024, torchvision native transforms). Predictions for YOLO and RT-DETR are exported via **predict** and therefore the resulting standardized AP numbers are **not expected to reproduce** `model.val()` exactly. Preserve native-replay results separately.

**To run:** Google Colab → Runtime → Change runtime type → T4 GPU → run all cells. Completed fold exports are saved progressively, so work survives interruptions. A ZIP is saved to Drive for sharing back to ChatGPT. Do not edit original files or tune settings on the held-out internal/external test images.

In [ ]:
%pip -q install "ultralytics==8.4.138" "torchmetrics>=1.3,<2" pycocotools scipy pandas pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os,sys,json,time,hashlib,platform,traceback,zipfile,gc
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import torch,torchvision,ultralytics,torchmetrics
from PIL import Image
from ultralytics import YOLO,RTDETR
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from scipy.stats import friedmanchisquare,wilcoxon
from itertools import combinations
from torchvision.models.detection import fasterrcnn_resnet50_fpn,retinanet_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

assert torch.cuda.is_available(), 'A CUDA GPU is required. Runtime > Change runtime type > T4 GPU.'
ROOT=Path('/content/drive/MyDrive/Pediatric_Caries_AI')
CV=ROOT/'cross_validation'
assert CV.is_dir(),CV
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=ROOT/'reviewer_revision'/f'Step_04C_Unified_COCO_AP_{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
DEV='cuda:0'; SEED=42; IMG_SIZE=1024; SCORE_FLOOR=.001; NMS_IOU=.70; MAX_DET=100
FAMILIES=['YOLO11s','RT-DETR-L','Faster R-CNN','RetinaNet']
EXPECTED_FOLD_GT=[50,48,31,49,28]
print('GPU:',torch.cuda.get_device_name(0),'Output:',OUT)
print('Torch:',torch.__version__,'Torchvision:',torchvision.__version__,'Ultralytics:',ultralytics.__version__,'TorchMetrics:',torchmetrics.__version__)
np.random.seed(SEED);torch.manual_seed(SEED)
run_meta={'utc':RUN_ID,'gpu':torch.cuda.get_device_name(0),'python':sys.version,'platform':platform.platform(),
          'torch':torch.__version__,'torchvision':torchvision.__version__,
          'ultralytics':ultralytics.__version__,'torchmetrics':torchmetrics.__version__,
          'scoring':{'framework':'TorchMetrics COCO MeanAveragePrecision pycocotools',
                     'max_detection_thresholds':[1,10,100],'metric_iou_levels':'0.50:0.95, step=0.05',
                     'score_floor':SCORE_FLOOR,'max_det':MAX_DET,'nms_iou_if_applicable':NMS_IOU,
                     'ultralytics_imgsz':IMG_SIZE,'batch_size':1,
                     'preprocessing':'family-native; not harmonized across training pipelines',
                     'ultralytics_inference':'predict (not val): may differ from native replay'} }
(OUT/'run_environment_and_protocol.json').write_text(json.dumps(run_meta,indent=2))

In [ ]:
# Fail before evaluation if any checkpoint has changed since the successful 04B provenance run.
EXPECTED_HASHES = {'YOLO11s|1': '0c8be9879987a156c08801fe7bcd72236010522b322af3bfe1796cd6a4ca4a58', 'RT-DETR-L|1': '22575dba1a9f5d32e37f3feb7a1389693223479ec88d81169c12d8f9deeca222', 'Faster R-CNN|1': 'fe6c3b4165bdfe872519a5ece501118846326050ccf1201fda68acff1303b084', 'RetinaNet|1': '7d73dc6177e5c19c1a59f02f0ea2c437e948043e2f378c6b0723405bf35649af', 'YOLO11s|2': '701c0e3841521c9ba6cd68a45bd86f40ac6e670bd7afb5b60bf8e321c3969403', 'RT-DETR-L|2': 'e6046bd4e64e4975ee8f7296fb30bbd724b481a88eb470f31e17ca702e7a9492', 'Faster R-CNN|2': '603bce3bdeca056834e3bd9e3f1d4885bb278a32ac06f054ea0d63ddfe288428', 'RetinaNet|2': 'f180d660f794ee45bb0bb672956c2126a93dc94bbe2e4dd47be9d642e0666c8f', 'YOLO11s|3': '5f45dd73709ebd7907db56ba1dcf9db294b0b71579f0e0486cb9a8b4b7f8e726', 'RT-DETR-L|3': '487f305c2fa3d372c5ce5b86c2f85f38ae8604d5cbfe8aaf213d1e4df4115192', 'Faster R-CNN|3': '9251577be1af6360302429210b13ff7611d6f7b26be9ba7ac37b0da5f2ea16e5', 'RetinaNet|3': '268e21a878b8db6f5da9c6721f6b7a91a354a8c9ddb10997367cbecad2d70ba1', 'YOLO11s|4': 'b280e7f979f1148dc04ca5a60138f3d60bebaed38991c92326d0e5c6ae601d27', 'RT-DETR-L|4': '17f4d986a8fa1c20dfb5c20b5bbc3cc7809fbbca73d19f2e36603910949f30f6', 'Faster R-CNN|4': 'd4801839dd7e2d691174e152317cfc6c6c4ceab08faa9d2cec427657a2724bf6', 'RetinaNet|4': '7a0ecc488c04f2175b9bbbb8a1ad183ad3e1fc79f64bce54a4b032129d3422ca', 'YOLO11s|5': '998e555d607e1203991578589d57e0c9369e13f2de6d3f116356e3e169441cb9', 'RT-DETR-L|5': 'd75949e566f5df8413498879a9576069530f0239e34bc1310cb6b173f7461a82', 'Faster R-CNN|5': '63ed7946ab05c0dc53708bcd0fabbaa2bf2b7d518c0281605eae807a8b44527f', 'RetinaNet|5': '0000c5eec1021cd575df66430e2e1f484737bc17ae1667812ee4460bb21666ca'}

def sha256(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(4*1024*1024),b''): h.update(b)
 return h.hexdigest()

def ckpt(fam,fold):
 if fam=='YOLO11s':return CV/f'models/YOLO11s/fold_{fold}/weights/best.pt'
 if fam=='RT-DETR-L':return CV/f'models/RTDETR/fold_{fold}/weights/best.pt'
 if fam=='Faster R-CNN':return CV/f'models/FasterRCNN/fold_{fold}/best.pt'
 if fam=='RetinaNet':return CV/f'models/RetinaNet/fold_{fold}/best.pt'
 raise ValueError(fam)

checks=[]
for fam in FAMILIES:
 for fold in range(1,6):
  p=ckpt(fam,fold)
  assert p.exists(),f'Missing: {p}'
  h=sha256(p)
  assert h==EXPECTED_HASHES[f'{fam}|{fold}'],f'CHECKPOINT CHANGED: {fam} fold={fold}: {h}'
  checks.append({'family':fam,'fold':fold,'relative_path':str(p.relative_to(ROOT)),'sha256':h,'size_bytes':p.stat().st_size})
pd.DataFrame(checks).to_csv(OUT/'checkpoint_sha256_verified.csv',index=False)
print('PASS: All 20 checkpoint hashes match Step 04B.')

In [ ]:
# Shared GT reader: same 14 images and YOLO normalized caries boxes for all four pipelines.
IMAGE_SUFFIXES={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
def fold_paths(fold):
 base=CV/f'fold_{fold}'
 images=sorted([p for p in (base/'images/val').iterdir() if p.suffix.lower() in IMAGE_SUFFIXES])
 assert len(images)==14,(fold,len(images))
 labeldir=base/'labels/val'
 return images,labeldir

def read_gt(imgpath,labeldir):
 with Image.open(imgpath) as im:
  W,H=im.size
 lp=labeldir/(imgpath.stem+'.txt')
 assert lp.is_file(),lp
 lines=[l.strip() for l in lp.read_text().splitlines() if l.strip()]
 boxes=[]
 for line in lines:
  fields=line.split()
  assert len(fields)==5,(lp,line)
  c,x,y,w,h=map(float,fields)
  assert int(c)==0 and all(np.isfinite([x,y,w,h])) and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1,(lp,line)
  xx1=(x-w/2)*W; yy1=(y-h/2)*H; xx2=(x+w/2)*W; yy2=(y+h/2)*H
  assert xx2>xx1 and yy2>yy1 and xx1>=-0.15 and yy1>=-0.15 and xx2<=W+0.15 and yy2<=H+0.15,(lp,line)
  boxes.append([max(0.,xx1),max(0.,yy1),min(float(W),xx2),min(float(H),yy2)])
 arr=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)
 return {'boxes':arr,'labels':torch.zeros((len(arr),),dtype=torch.int64)}, W,H,lp

man=[]
for f in range(1,6):
 images,labels=fold_paths(f)
 ng=0
 for ip in images:
  gt,W,H,lp=read_gt(ip,labels)
  ng+=len(gt['boxes'])
  man.append({'fold':f,'image':ip.name,'W':W,'H':H,'n_gt':len(gt['boxes']),
             'image_sha256':sha256(ip),'label_sha256':sha256(lp)})
 assert ng==EXPECTED_FOLD_GT[f-1],f'Fold {f}: expected {EXPECTED_FOLD_GT[f-1]} lesions, found {ng}'
assert len(man)==70
pd.DataFrame(man).to_csv(OUT/'fold_gt_manifest_sha256.csv',index=False)
print('PASS: 70 held-out validation image instances, 206 source-defined caries boxes.')

In [ ]:
# Prediction generators. Thresholds and output cap harmonized as far as model APIs allow.
def build_torchvision_model(fam):
 if fam=='Faster R-CNN':
  m=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None)
  n=m.roi_heads.box_predictor.cls_score.in_features
  m.roi_heads.box_predictor=FastRCNNPredictor(n,2)
  m.roi_heads.score_thresh=SCORE_FLOOR
  m.roi_heads.nms_thresh=NMS_IOU
  m.roi_heads.detections_per_img=MAX_DET
 elif fam=='RetinaNet':
  m=retinanet_resnet50_fpn(weights=None,weights_backbone=None)
  m.head.classification_head=RetinaNetClassificationHead(m.backbone.out_channels,m.head.classification_head.num_anchors,2)
  m.score_thresh=SCORE_FLOOR
  m.nms_thresh=NMS_IOU
  m.detections_per_img=MAX_DET
 else: raise ValueError(fam)
 return m

def load_model(fam,fold):
 p=ckpt(fam,fold)
 if fam in ['YOLO11s','RT-DETR-L']:
  model=(YOLO if fam=='YOLO11s' else RTDETR)(str(p))
  assert model.names=={0:'caries'},(fam,model.names)
  return model
 m=build_torchvision_model(fam)
 state=torch.load(str(p),map_location='cpu',weights_only=True)
 if isinstance(state,dict) and 'state_dict' in state:state=state['state_dict']
 m.load_state_dict(state,strict=True)
 del state
 return m.eval().to(DEV)

@torch.inference_mode()
def predict_one(fam,model,ip,W,H):
 if fam in ['YOLO11s','RT-DETR-L']:
  results=model.predict(source=str(ip),device=0,imgsz=IMG_SIZE,conf=SCORE_FLOOR,
                         iou=NMS_IOU,max_det=MAX_DET,rect=False,half=False,
                         augment=False,save=False,verbose=False,stream=False)
  assert len(results)==1,(fam,ip,len(results))
  bb=results[0].boxes
  if bb is None:
   box=torch.empty((0,4),dtype=torch.float32)
   scores=torch.empty((0,),dtype=torch.float32)
   labels=torch.empty((0,),dtype=torch.long)
  else:
   box=bb.xyxy.detach().cpu().to(torch.float32)
   scores=bb.conf.detach().cpu().to(torch.float32)
   labels=bb.cls.detach().cpu().to(torch.int64)
 else:
  im=Image.open(ip).convert('RGB')
  tens=torchvision.transforms.functional.to_tensor(im).to(DEV)
  im.close()
  out=model([tens])[0]
  box=out['boxes'].detach().cpu().to(torch.float32)
  scores=out['scores'].detach().cpu().to(torch.float32)
  labels=out['labels'].detach().cpu().to(torch.int64)
  assert torch.all(labels==1),(fam,ip,torch.unique(labels))
  labels=torch.zeros_like(labels)
 # Require predictable clean and standardized outputs (no silent discarding).
 assert box.ndim==2 and box.shape[1]==4 and len(box)==len(scores)==len(labels)
 assert torch.isfinite(box).all() and torch.isfinite(scores).all()
 assert torch.all(labels==0),(fam,ip,torch.unique(labels))
 inds=(scores>=SCORE_FLOOR).nonzero().reshape(-1)
 inds=inds[torch.argsort(scores[inds],descending=True,stable=True)][:MAX_DET]
 box=box[inds].clone();scores=scores[inds].clone();labels=labels[inds].clone()
 box[:,[0,2]]=box[:,[0,2]].clamp(0,W)
 box[:,[1,3]]=box[:,[1,3]].clamp(0,H)
 assert torch.all(box[:,2]>box[:,0]) and torch.all(box[:,3]>box[:,1]),(fam,ip,'Invalid prediction box')
 return {'boxes':box,'scores':scores,'labels':labels}

print('Configured common COCO evaluator and model-specific output adapters.')

In [ ]:
# Run 20 folds; save per-image predictions and metrics after EVERY fold.
# If interrupted, previous results remain in Google Drive. Share the partial folder/ZIP for diagnosis.
METRIC_COLS=['family','fold','COCO_mAP50','COCO_mAP50_95','COCO_mAR100','n_images','n_gt','n_predictions','status','error','duration_seconds']
rows=[];failures=[]
for fam in FAMILIES:
 for fold in range(1,6):
  started=time.time();row={'family':fam,'fold':fold,'status':'ERROR','error':''}
  model=None;metric=None
  try:
   ims,labdir=fold_paths(fold)
   model=load_model(fam,fold)
   metric=MeanAveragePrecision(box_format='xyxy',iou_type='bbox',backend='pycocotools',
                               max_detection_thresholds=[1,10,100],class_metrics=False)
   pred_rows=[];num_gt=0;num_pr=0
   for ip in ims:
    gt,W,H,lp=read_gt(ip,labdir)
    pdict=predict_one(fam,model,ip,W,H)
    metric.update([pdict],[gt])
    num_gt+=len(gt['boxes']);num_pr+=len(pdict['boxes'])
    if len(pdict['boxes'])==0:
     pred_rows.append({'family':fam,'fold':fold,'image':ip.name,'prediction_index':-1,
                       'x1':np.nan,'y1':np.nan,'x2':np.nan,'y2':np.nan,'score':np.nan,'label':np.nan})
    for i,(b,s,c) in enumerate(zip(pdict['boxes'].tolist(),pdict['scores'].tolist(),pdict['labels'].tolist())):
     pred_rows.append({'family':fam,'fold':fold,'image':ip.name,'prediction_index':i,
                       'x1':b[0],'y1':b[1],'x2':b[2],'y2':b[3],'score':s,'label':int(c)})
   assert num_gt==EXPECTED_FOLD_GT[fold-1]
   rr=metric.compute()
   vals={k:float(rr[k]) for k in ['map_50','map','mar_100']}
   assert all(0<=v<=1 for v in vals.values()),(fam,fold,vals)
   row.update({'COCO_mAP50':vals['map_50'],'COCO_mAP50_95':vals['map'],
               'COCO_mAR100':vals['mar_100'],'n_images':14,'n_gt':num_gt,'n_predictions':num_pr,
               'status':'OK','error':''})
   # Save predictions even for folds with no predicted boxes.
   pd.DataFrame(pred_rows).to_csv(OUT/f'predictions_{fam.replace(" ","_")}_fold{fold}.csv',index=False)
   print('PASS',fam,'fold',fold,'mAP50',round(vals['map_50'],5),
         'mAP50:95',round(vals['map'],5),'nPred',num_pr)
  except Exception as exc:
   row.update({'status':'ERROR','error':repr(exc)})
   failures.append((fam,fold,repr(exc)))
   print('ERROR',fam,'fold',fold,repr(exc))
   traceback.print_exc(limit=3)
  row['duration_seconds']=round(time.time()-started,1)
  rows.append(row)
  pd.DataFrame(rows).reindex(columns=METRIC_COLS).to_csv(OUT/'unified_coco_per_fold.csv',index=False)
  del metric,model
  gc.collect();torch.cuda.empty_cache()
print('Completed',len(rows),'success',sum(r['status']=='OK' for r in rows),'failures',len(failures))

In [ ]:
# A secondary statistical summary from standardized-AP fold-wise results.
metrics=pd.read_csv(OUT/'unified_coco_per_fold.csv')
assert len(metrics)==20 and (metrics.status=='OK').all(), 'STOP: some folds failed; do not infer full results from partial data.'
summary=[];testrows=[];pairrows=[]
for m in ['COCO_mAP50','COCO_mAP50_95']:
 wide=metrics.pivot(index='fold',columns='family',values=m)[FAMILIES]
 assert wide.shape==(5,4) and not wide.isna().any().any()
 for fam in FAMILIES:
  y=wide[fam]
  summary.append({'model':fam,'metric':m,'n_folds':5,'mean':y.mean(),'sample_SD':y.std(ddof=1),
                  'min':y.min(),'max':y.max()})
 fr=friedmanchisquare(*(wide[fam] for fam in FAMILIES))
 testrows.append({'metric':m,'friedman_chi2':float(fr.statistic),
                  'friedman_p_asymptotic':float(fr.pvalue),'n_folds':5})
 local=[]
 for a,b in combinations(FAMILIES,2):
  w=wilcoxon(wide[a],wide[b],alternative='two-sided',zero_method='wilcox',method='auto')
  local.append({'metric':m,'model_a':a,'model_b':b,'W':float(w.statistic),'p_raw':float(w.pvalue)})
 # Holm step-down in predeclared 6 tests for each endpoint; enforce monotonicity.
 ordered=sorted(range(len(local)),key=lambda k:local[k]['p_raw'])
 last=0.
 for rank,idx in enumerate(ordered):
  candidate=min(1.,(len(local)-rank)*local[idx]['p_raw'])
  last=max(last,candidate)
  local[idx]['p_Holm']=last
  local[idx]['significant_at_0_05']=bool(last<.05)
 pairrows+=local
pd.DataFrame(summary).to_csv(OUT/'unified_coco_mean_sd.csv',index=False)
pd.DataFrame(testrows).to_csv(OUT/'unified_coco_friedman.csv',index=False)
pd.DataFrame(pairrows).to_csv(OUT/'unified_coco_wilcoxon_holm.csv',index=False)
print(pd.DataFrame(summary).to_string(index=False))
print(pd.DataFrame(testrows).to_string(index=False))
print('Pairwise Holm-adjusted significant:',sum(x['significant_at_0_05'] for x in pairrows))

In [ ]:
# Readme and shareable ZIP (no raw copyrighted radiograph pixels are redistributed).
notes='STEP 04C / UNIFIED COCO AP CONTROL — EVIDENCE BUNDLE\nOne fixed checkpoint per model/fold, same 14 validation images/fold and 206 total source-defined lesions.\nThe unified evaluator is TorchMetrics pycocotools; scores use a 0.001 floor and at most 100 predictions/image.\nUltralytics uses 1024-pixel predictor output, while torchvision uses native full-resolution model transforms.\nThe models were trained using different configurations. This control harmonizes evaluation metrics, not optimization budgets.\nUltralytics predict versus val can give different prediction sets; COCO AP here is NOT required to match the Step 04B native model.val() results.\nThe original GT is used as annotated; expert comments suggest annotation limitations. This is not a corrected-reference experiment.\nResults based on only five overlapping CV training folds are exploratory; do not claim pairwise superiority without support.\nThe script checks Step 04B checkpoint SHA256, image-label pairing and fold lesion counts, and saves per-image output.\n'
(OUT/'README_LIMITATIONS.txt').write_text(notes)
zip_path=OUT/'Step_04C_Unified_COCO_AP_Results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
 for file in sorted(OUT.iterdir()):
  if file.is_file() and file!=zip_path:z.write(file,file.name)
print('RESULTS:',zip_path)
print('ZIP bytes:',zip_path.stat().st_size)
print('Please upload the ZIP here for independent review. DO NOT update Table 2 until numbers are reviewed.')